# 01 Run Orchestrator

역할: runtime과 dataset 상태를 맞추고, baseline/category/manifest를 선택해 runner를 실행한다.
산출물은 `summary.json`과 `log.txt` handoff까지만 확인한다.


# Condition Shift Baseline Run Notebook

이 노트북에서 하는 일:

- Colab/서버에서 Git checkout과 dataset 준비 상태를 맞춘다.
- `PatchCore`, `UniVAD` runner를 `single` 또는 `sequential` 모드로 실행한다.
- 실행 결과로 남는 `summary.json`, `log.txt` 경로와 존재 여부를 handoff table로 확인한다.

중요한 순서:

1. Drive mount
2. Git checkout 또는 pull
3. checkout된 repo 기준으로 path/helper import
4. dataset/bootstrap/setup/run

분석과 figure export는 각각 `02_analysis_dashboard.ipynb`, `03_figure_export.ipynb`에서 수행한다.


## Cell Role: Drive Mount

Colab에서 Drive를 mount한다. 로컬 Jupyter에서는 `google.colab`이 없으므로 이 셀은 자동으로 skip된다.
Drive는 dataset tar, checkpoint backup, mask backup 같은 runtime 외부 자산 접근에만 사용한다.


In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('google.colab is not available; skipping Drive mount for local runtime.')


Mounted at /content/drive


## Cell Role: Git Checkout

runner/helper를 import하기 전에 `/content/ReGraM`을 clone 또는 pull 한다.
이 순서를 지켜야 Colab 런타임에 남아 있던 예전 코드가 먼저 import되는 충돌을 피할 수 있다.


In [2]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/outSeop/ReGraM.git'
cwd = Path.cwd().resolve()
local_repo = next((p.resolve() for p in [cwd, *cwd.parents] if p.exists() and p.name == 'ReGraM'), None)

if Path('/content').exists():
    REPO_DIR = Path('/content/ReGraM')
    git_cmd = (
        f'if [ -d "{REPO_DIR}/.git" ]; then git -C "{REPO_DIR}" pull --ff-only; '
        f'else git clone "{REPO_URL}" "{REPO_DIR}"; fi'
    )
    print(git_cmd)
    subprocess.run(['bash', '-lc', git_cmd], check=True)
    CHECKOUT_ROOT = REPO_DIR.resolve()
else:
    CHECKOUT_ROOT = local_repo or cwd
    print('local runtime detected; skipping Colab clone/pull')

print('checkout root =', CHECKOUT_ROOT)


if [ -d "/content/ReGraM/.git" ]; then git -C "/content/ReGraM" pull --ff-only; else git clone "https://github.com/outSeop/ReGraM.git" "/content/ReGraM"; fi
checkout root = /content/ReGraM


## Cell Role: Path and Helper Setup

방금 checkout된 repo를 기준으로 `REPO_ROOT`, `EXP_ROOT`, `REPORT_ROOT`를 확정하고 Python helper를 import한다.
이 셀 이후의 모든 경로와 runner command는 여기서 확정된 root를 따른다.


In [ ]:
from pathlib import Path
import importlib
import subprocess
import sys

import pandas as pd
import torch
from IPython.display import display

colab_checkout = Path('/content/ReGraM')
cwd = Path.cwd().resolve()
repo_candidates = [colab_checkout, cwd, *cwd.parents]
REPO_ROOT = next((p.resolve() for p in repo_candidates if p.exists() and p.name == 'ReGraM'), cwd)
EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
REPORT_ROOT = EXP_ROOT / 'reports'
NOTEBOOK_ROOT = EXP_ROOT / 'notebook'
MANIFEST_ROOT_CANDIDATES = [REPO_ROOT / 'manifests', EXP_ROOT / 'manifests']
SRC_ROOT = EXP_ROOT / 'src'
CORE_SRC = SRC_ROOT / 'core'
UNIVAD_SRC = SRC_ROOT / 'univad'
for source_root in (SRC_ROOT, CORE_SRC, UNIVAD_SRC):
    if str(source_root) not in sys.path:
        sys.path.insert(0, str(source_root))

# Colab/Jupyter keeps imported modules alive after git pull. Reload notebook helpers
# so rerunning this cell picks up the current checkout without a full runtime restart.
for module_name in (
    'orchestration.notebook_orchestration',
    'notebook_orchestration',
    'setup_runtime',
):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

from orchestration.notebook_orchestration import (
    build_baseline_specs,
    build_requested_run_configs,
    discover_manifest_names,
    display_environment_summary,
    display_experiment_preset,
    display_run_plan,
    display_title,
    load_experiment_config,
    model_config_from_experiment_config,
    model_sweep_from_experiment_config,
    resolve_manifest_paths,
    resolve_requested_baselines,
    run_execution_queue,
    validate_baseline_names,
)
from setup_runtime import default_univad_setup_flags, evaluate_baseline_readiness, run_baseline_setup

DEFAULT_PATCHCORE_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
run_history = []

display_environment_summary(REPO_ROOT, EXP_ROOT, REPORT_ROOT)
print('REPO_ROOT =', REPO_ROOT)
print('EXP_ROOT =', EXP_ROOT)


## Cell Role: Raw Dataset Load

Drive에 있는 raw MVTec LOCO tar를 Colab runtime의 repo data 경로로 복사/해제한다.
이미 `data/row/mvtec_loco_anomaly_detection`이 있으면 재해제하지 않는다.


In [ ]:
from pathlib import Path
import shutil
import subprocess

drive_tar = Path("/content/drive/MyDrive/ReGraM/data/row/mvtec_loco_anomaly_detection.tar.gz")
runtime_tar = Path("/content/mvtec_loco_anomaly_detection.tar.gz")
runtime_row = REPO_ROOT / "data" / "row"
runtime_root = runtime_row / "mvtec_loco_anomaly_detection"


def find_nested_dataset_root(base_dir: Path, dataset_name: str):
    direct = base_dir / dataset_name
    if direct.exists():
        return direct
    matches = [path for path in base_dir.rglob(dataset_name) if path.is_dir()]
    if not matches:
        return None
    matches = sorted(matches, key=lambda path: len(path.parts))
    return matches[0]


def normalize_dataset_root(base_dir: Path, dataset_name: str):
    target_root = base_dir / dataset_name
    discovered_root = find_nested_dataset_root(base_dir, dataset_name)
    if discovered_root is None:
        return target_root, False
    if discovered_root == target_root:
        return target_root, False
    print(f"normalize dataset root: {discovered_root} -> {target_root}")
    target_root.parent.mkdir(parents=True, exist_ok=True)
    if target_root.exists():
        raise RuntimeError(f"target dataset root already exists: {target_root}")
    shutil.move(str(discovered_root), str(target_root))
    return target_root, True


print("drive_tar exists:", drive_tar.exists())
print("runtime_root exists:", runtime_root.exists())

runtime_row.mkdir(parents=True, exist_ok=True)

if not runtime_root.exists():
    if not runtime_tar.exists():
        subprocess.run(["cp", str(drive_tar), str(runtime_tar)], check=True)
    subprocess.run(
        ["tar", "-xf", str(runtime_tar), "-C", str(runtime_row)],
        check=True,
    )

runtime_root, normalized = normalize_dataset_root(runtime_row, "mvtec_loco_anomaly_detection")
print("normalized:", normalized)
print("done")
print(runtime_root.exists(), runtime_root)


## Cell Role: Optional Dataset Bootstrap

prepared dataset, small report artifact, 기타 보조 자산을 별도 bootstrap script로 확인한다.
현재는 `--dry-run`이므로 실제 복사 없이 무엇이 필요한지만 점검한다.


In [ ]:
bootstrap_cmd = [
    sys.executable,
    str(EXP_ROOT / 'colab' / 'bootstrap_runtime.py'),
    '--dry-run',
]
print(' '.join(bootstrap_cmd))
subprocess.run(bootstrap_cmd, cwd=REPO_ROOT, check=True)


## Cell Role: Experiment Scope Config

이번 실행의 baseline, category, manifest 범위, wandb 설정, readiness 정책을 결정한다.
모델 하이퍼파라미터와 sweep은 `EXPERIMENT_CONFIG_PATH`가 가리키는 YAML의 `models` / `sweep` 섹션을 읽는다.
`CONFIGURED_MANIFEST_NAMES`가 비어 있지 않으면 그 목록만 실행하고, 전체 auto-discovery는 쓰지 않는다.
`SELECTED_SEVERITIES`가 비어 있으면 low/medium/high 전체를 실행하고, 값이 있으면 해당 severity만 실행한다.


In [ ]:
RUN_MODE = 'sequential'  # choose 'single' or 'sequential'
RUN_BASELINE = ['UniVAD', 'PatchCore']      # used only when RUN_MODE == 'single'
SEQUENTIAL_BASELINES = ['UniVAD', 'PatchCore']
DASHBOARD_BASELINES = ['UniVAD', 'PatchCore']
# breakfast_box, juice_bottle, screw_bag, pushpins, splicing_connectors
CATEGORIES = ['breakfast_box', 'screw_bag']
AUTO_DISCOVER_MANIFESTS = False
CONFIGURED_MANIFEST_NAMES = ['query_gaussian_noise.jsonl', 'query_gaussian_blur.jsonl', 'query_position_shift.jsonl']
SELECTED_SEVERITIES = ['high']  # [] means all; examples: ['low'], ['medium'], ['high'], ['low', 'high']
EXCLUDED_MANIFEST_NAMES = {'query_identity.jsonl', 'query_multi.jsonl'}
WANDB_PROJECT = 'regram-condition-shift-2'
WANDB_MODE = 'online'
SHOW_RUNNER_OUTPUT = True
RUNNER_OUTPUT_TAIL_LINES = 40
RUN_ONLY_READY_BASELINES = True
STOP_ON_FAILURE = True
TOP_K_WORST_SHIFTS = 10
EXPERIMENT_CONFIG_PATH = EXP_ROOT / 'configs' / 'experiment_template.yaml'

try:
    load_experiment_config
except NameError:
    import orchestration.notebook_orchestration as notebook_orchestration_module
    importlib.reload(notebook_orchestration_module)
    from orchestration.notebook_orchestration import (
        load_experiment_config,
        model_config_from_experiment_config,
        model_sweep_from_experiment_config,
    )

EXPERIMENT_CONFIG = load_experiment_config(EXPERIMENT_CONFIG_PATH)
MODEL_CONFIG = model_config_from_experiment_config(EXPERIMENT_CONFIG)
MODEL_SWEEP = model_sweep_from_experiment_config(EXPERIMENT_CONFIG)

RAW_LOCO_ROOT = REPO_ROOT / 'data' / 'row' / 'mvtec_loco_anomaly_detection'
UNIVAD_CAPTION_DATA_ROOT = REPO_ROOT / 'data' / 'mvtec_loco_caption'
UNIVAD_SETUP_FLAGS = default_univad_setup_flags()

BASELINE_SPECS = build_baseline_specs(
    repo_root=REPO_ROOT,
    exp_root=EXP_ROOT,
    raw_loco_root=RAW_LOCO_ROOT,
    univad_caption_data_root=UNIVAD_CAPTION_DATA_ROOT,
    default_patchcore_device=DEFAULT_PATCHCORE_DEVICE,
    model_config=MODEL_CONFIG,
    model_sweep=MODEL_SWEEP,
)
validate_baseline_names(DASHBOARD_BASELINES, specs=BASELINE_SPECS, label='dashboard')
REQUESTED_BASELINES = resolve_requested_baselines(
    RUN_MODE,
    RUN_BASELINE,
    SEQUENTIAL_BASELINES,
    specs=BASELINE_SPECS,
)
ACTIVE_MANIFEST_NAMES = discover_manifest_names(
    manifest_roots=MANIFEST_ROOT_CANDIDATES,
    configured_names=CONFIGURED_MANIFEST_NAMES,
    auto_discover=AUTO_DISCOVER_MANIFESTS,
    excluded_names=EXCLUDED_MANIFEST_NAMES,
)
manifest_paths = resolve_manifest_paths(ACTIVE_MANIFEST_NAMES, MANIFEST_ROOT_CANDIDATES)
requested_run_configs = build_requested_run_configs(
    active_baselines=REQUESTED_BASELINES,
    specs=BASELINE_SPECS,
    categories=CATEGORIES,
    manifest_paths=manifest_paths,
    manifest_names=ACTIVE_MANIFEST_NAMES,
    report_root=REPORT_ROOT,
    wandb_project=WANDB_PROJECT,
    wandb_mode=WANDB_MODE,
    selected_severities=SELECTED_SEVERITIES,
)
dashboard_run_configs = build_requested_run_configs(
    active_baselines=DASHBOARD_BASELINES,
    specs=BASELINE_SPECS,
    categories=CATEGORIES,
    manifest_paths=manifest_paths,
    manifest_names=ACTIVE_MANIFEST_NAMES,
    report_root=REPORT_ROOT,
    wandb_project=WANDB_PROJECT,
    wandb_mode=WANDB_MODE,
    selected_severities=SELECTED_SEVERITIES,
)
baseline_status_df = evaluate_baseline_readiness(REQUESTED_BASELINES, specs=BASELINE_SPECS, categories=CATEGORIES)
run_configs = list(requested_run_configs)
skipped_run_configs = []

display_experiment_preset(
    run_mode=RUN_MODE,
    requested_baselines=REQUESTED_BASELINES,
    dashboard_baselines=DASHBOARD_BASELINES,
    categories=CATEGORIES,
    manifest_names=ACTIVE_MANIFEST_NAMES,
    wandb_mode=WANDB_MODE,
    stop_on_failure=STOP_ON_FAILURE,
    univad_flags=UNIVAD_SETUP_FLAGS,
    specs=BASELINE_SPECS,
    requested_run_configs=requested_run_configs,
)


## Cell Role: Run Matrix Preview

실제로 실행될 baseline/category/manifest 수와 runner command를 실행 전에 확인한다.
여기서 manifest count가 예상과 다르면 아래 실행 셀로 내려가지 않는다.


In [ ]:
display_title('Current Run Matrix', 'Single mode executes one baseline. Sequential mode executes the listed baselines in order and keeps a shared comparison dashboard.')
command_preview_df = pd.DataFrame([
    {
        'baseline': config['baseline'],
        'category': config['category'],
        'manifest_count': len(config['manifest_paths']),
        'severities': ', '.join(config.get('selected_severities', [])) or 'all',
        'device': config['device'],
        'summary_name': config['summary_path'].name,
        'log_name': config['log_path'].name,
        'runner_cmd': ' '.join(config['runner_cmd']),
    }
    for config in requested_run_configs
])
display(command_preview_df)


## Cell Role: Baseline Setup and Readiness

외부 repo, dependency, checkpoint, dataset, grounding mask 상태를 baseline별로 점검하고 필요한 setup을 수행한다.
`RUN_ONLY_READY_BASELINES`가 켜져 있으면 준비되지 않은 run은 execution queue에서 제외한다.


In [ ]:
display_title(
    'Baseline Setup & Readiness',
    'Setup/check external repos, runtime dependencies, checkpoints, datasets, and masks before execution.',
)
setup_result = run_baseline_setup(
    requested_baselines=REQUESTED_BASELINES,
    specs=BASELINE_SPECS,
    categories=CATEGORIES,
    raw_loco_root=RAW_LOCO_ROOT,
    settings=UNIVAD_SETUP_FLAGS,
    requested_run_configs=requested_run_configs,
    run_only_ready_baselines=RUN_ONLY_READY_BASELINES,
)
setup_df = setup_result['setup_df']
baseline_status_df = setup_result['baseline_status_df']
run_configs = setup_result['run_configs']
skipped_run_configs = setup_result['skipped_run_configs']

with pd.option_context('display.max_colwidth', None, 'display.max_rows', 80):
    display(setup_df)
if 'UniVAD' in REQUESTED_BASELINES:
    for title, key in [
        ('UniVAD Dataset Status', 'univad_dataset_status_df'),
        ('UniVAD Local Dependency Status', 'univad_local_dependency_status_df'),
        ('UniVAD Checkpoint Status', 'univad_checkpoint_status_df'),
        ('UniVAD Grounding Mask Status', 'univad_mask_status_df'),
        ('UniVAD Missing Mask Preview', 'univad_missing_mask_preview_df'),
    ]:
        display_title(title)
        with pd.option_context('display.max_colwidth', None, 'display.max_rows', 80):
            display(setup_result[key])

display_title('Baseline Readiness')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', 80):
    display(baseline_status_df)
if skipped_run_configs:
    display_title('Skipped Runs')
    status_by_baseline = baseline_status_df.set_index('baseline').to_dict('index') if not baseline_status_df.empty else {}
    skipped_df = pd.DataFrame([
        {
            'baseline': config['baseline'],
            'category': config['category'],
            'blockers': status_by_baseline.get(config['baseline'], {}).get('blockers', '-'),
            'missing_mask_paths': status_by_baseline.get(config['baseline'], {}).get('missing_mask_paths', '-'),
            'summary_path': str(config['summary_path']),
            'log_path': str(config['log_path']),
        }
        for config in skipped_run_configs
    ])
    with pd.option_context('display.max_colwidth', None, 'display.max_rows', 80):
        display(skipped_df)

display_title('Execution Queue', f'Prepared `{len(run_configs)}` runnable jobs.')
display_run_plan(run_configs)


## Cell Role: Execute Run Queue

준비가 끝난 run queue를 순서대로 실행한다.
이 셀은 오래 걸릴 수 있고, runner output은 `SHOW_RUNNER_OUTPUT` / `RUNNER_OUTPUT_TAIL_LINES` 설정을 따른다.


In [ ]:
run_history = run_execution_queue(
    run_configs=run_configs,
    repo_root=REPO_ROOT,
    show_runner_output=SHOW_RUNNER_OUTPUT,
    runner_output_tail_lines=RUNNER_OUTPUT_TAIL_LINES,
    stop_on_failure=STOP_ON_FAILURE,
)


## Cell Role: Output Handoff

runner가 남긴 `summary.json`과 `log.txt`의 존재 여부와 경로만 확인한다.
결과 해석, 비교 테이블, 시각화는 `02_analysis_dashboard.ipynb`에서 수행한다.


In [ ]:
summary_sources = dashboard_run_configs if 'dashboard_run_configs' in globals() else requested_run_configs
handoff_rows = []
for config in summary_sources:
    summary_path = config['summary_path']
    log_path = config['log_path']
    handoff_rows.append(
        {
            'baseline': config['baseline'],
            'category': config['category'],
            'summary_exists': summary_path.exists(),
            'summary_path': str(summary_path),
            'log_exists': log_path.exists(),
            'log_path': str(log_path),
        }
    )

display_title(
    'Output Handoff',
    'Open notebook/02_analysis_dashboard.ipynb for tables, plots, and interpretation.',
)
display(pd.DataFrame(handoff_rows))
